 Étape 1 complète

In [7]:
import numpy as np
import pandas as pd
from scipy import stats
import random
from datetime import datetime, timedelta
from faker import Faker


In [8]:
print(random.seed(42))

None


In [11]:

# ── Reproductibilité ─────────────────────────────────────────────────────────
random.seed(42)
np.random.seed(42)
fake = Faker()
Faker.seed(42)

# ── Paramètres ──────────────────────────────────────────────────────────────
NB_USERS = 500
INTERESTS = ["fitness", "musique", "technologie", "cuisine", "voyage", "cinema"]
INTEREST_PROBS = [0.25, 0.22, 0.18, 0.15, 0.12, 0.08]

ACTIONS = {
    "fitness":     ["watched workout video", "bought protein powder",
                    "liked gym post", "joined gym"],
    "musique":     ["watched music video", "bought headphones",
                    "liked song", "attended concert", "streamed album"],
    "technologie": ["watched AI talk", "read tech blog", "bought laptop",
                    "bought smartwatch", "liked AI article",
                    "contributed to open source"],
    "cuisine":     ["watched cooking show", "bought ingredients",
                    "liked recipe post", "tried new recipe",
                    "bought kitchen gadget"],
    "voyage":      ["watched travel vlog", "booked flight", "bought luggage",
                    "liked travel photo", "checked into hotel"],
    "cinema":      ["watched movie", "watched trailer", "bought streaming sub",
                    "rated film", "bought cinema ticket"],
}

# ── Inférence du type d'action ───────────────────────────────────────────────
def infer_action_type(action):
    """Déduit le type d'action à partir du verbe."""
    if isinstance(action, str):
        if action.startswith(("watched", "read", "streamed")):
            return "view"
        if action.startswith("bought") or action == "booked flight":
            return "buy"
        if action.startswith("liked") or action == "rated film":
            return "like"
    return "other"

# ── Génération des profils ────────────────────────────────
# ── Génération des profils ────────────────────────────────
def generate_users(n=NB_USERS):
    profiles = []
    for i in range(n):
        age = int(np.clip(np.random.normal(loc=35, scale=10), 18, 65))
        nb_interests = random.randint(1, 6)
        interests = list(np.random.choice(
            INTERESTS, size=nb_interests, replace=False, p=INTEREST_PROBS
        ))
        # Générer l'activity_log directement dans le profil
        activity_log = []
        for _ in range(random.randint(5, 15)):
            # 80% du temps : activité dans ses intérêts
            # 20% du temps : activité hors intérêts (comme dans la réalité)
            if random.random() < 0.80:
                category = random.choice(interests)
            else:
                other = [c for c in INTERESTS if c not in interests]
                if other:
                    category = random.choice(other)
                else:
                    category = random.choice(interests)
            activity_log.append(random.choice(ACTIONS[category]))

        profiles.append({
            "name":         fake.name(),
            "age":          age,
            "interests":    interests,
            "activity_log": activity_log,
        })
    return profiles

# ── Conversion en DataFrames pour l'analyse ─────────────────────────────────
def profiles_to_dataframes(profiles):
    """Convertit les profils en 2 DataFrames : users + logs."""
    user_records = []
    log_records  = []
    base_date = datetime(2024, 1, 1)

    for idx, profile in enumerate(profiles):
        # DataFrame utilisateurs
        user_records.append({
            "user_id":   idx + 1,
            "name":      profile["name"],
            "age":       profile["age"],
            "interests": profile["interests"],
        })

        # DataFrame logs
        for action in profile["activity_log"]:
            action_type = infer_action_type(action)
            # Déduire la catégorie de l'action
            category = None
            for cat, actions in ACTIONS.items():
                if action in actions:
                    category = cat
                    break

            delta = timedelta(days=random.randint(0, 180),
                              hours=random.randint(0, 23))
            ts = base_date + delta
            log_records.append({
                "user_id":     idx + 1,
                "action":      action,
                "action_type": action_type,
                "category":    category,
                "timestamp":   ts,
                "hour":        ts.hour,
            })

    users_df = pd.DataFrame(user_records)
    logs_df  = pd.DataFrame(log_records)
    return users_df, logs_df

# ── Injection de bruit (doublons + NaN) ──────────────────────────────────────
def inject_noise(df, dup_fraction=0.03, nan_fraction=0.02):
    """Injecte des doublons et des NaN pour simuler des données réelles."""
    df_noisy = df.copy()

    # ── Doublons ──
    n_dup = int(len(df_noisy) * dup_fraction)
    dup_rows = df_noisy.sample(n=n_dup, random_state=42)
    df_noisy = pd.concat([df_noisy, dup_rows], ignore_index=True)

    # ── Valeurs manquantes ──
    n_nan = int(len(df_noisy) * nan_fraction)
    nan_indices = df_noisy.sample(n=n_nan, random_state=42).index
    for idx in nan_indices:
        col = random.choice(["action", "category", "hour"])
        df_noisy.loc[idx, col] = np.nan

    return df_noisy

# ── Exécution ────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # 1. Générer les profils (format dict comme le sujet)
    profiles = generate_users()

    # 2. Afficher un profil exemple
    print("=== Exemple de profil ===")
    print(profiles[0])

    # 3. Convertir en DataFrames pour l'analyse
    users_df, logs_df = profiles_to_dataframes(profiles)
    logs_df = inject_noise(logs_df)

    print("\n=== DataFrame Utilisateurs ===")
    print(users_df)
    print(f"\nShape : {users_df.shape}")

    print("\n=== DataFrame Logs ===")
    print(logs_df.head())
    print(f"\nShape : {logs_df.shape}")
    print(f"Valeurs manquantes :\n{logs_df.isnull().sum()}")
    print(f"Doublons : {logs_df.duplicated().sum()}")

=== Exemple de profil ===
{'name': 'Allison Hill', 'age': 39, 'interests': [np.str_('cuisine'), np.str_('technologie'), np.str_('fitness'), np.str_('voyage'), np.str_('musique'), np.str_('cinema')], 'activity_log': ['bought protein powder', 'watched movie', 'watched music video', 'watched cooking show', 'liked AI article', 'bought headphones']}

=== DataFrame Utilisateurs ===
     user_id                 name  age  \
0          1         Allison Hill   39   
1          2          Noah Rhodes   33   
2          3      Angie Henderson   24   
3          4        Daniel Wagner   38   
4          5      Cristian Santos   28   
..       ...                  ...  ...   
495      496  Lawrence Harrington   39   
496      497       Austin Osborne   34   
497      498        James Bradley   41   
498      499          Meghan Rush   19   
499      500        Phillip Berry   31   

                                             interests  
0    [cuisine, technologie, fitness, voyage, musiqu...  
1 

 Étape 2 complète

In [13]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("=== ÉTAPE 2 : PRÉTRAITEMENT ET ANALYSE STATISTIQUE ===")
print("="*60)

# ============================================
# 1. NETTOYAGE DES DONNÉES
# ============================================
print("\n1️⃣  NETTOYAGE DES DONNÉES")
print("-"*40)

print(f"📊 Avant nettoyage:")
print(f"   - Logs: {len(logs_df)} lignes")
print(f"   - Utilisateurs: {len(users_df)} lignes")

# 1.1 Supprimer les doublons
logs_clean = logs_df.drop_duplicates()
duplicates_removed = len(logs_df) - len(logs_clean)
print(f"\n✅ Doublons supprimés: {duplicates_removed}")

# 1.2 Gérer les valeurs manquantes
nan_before = logs_clean.isnull().sum().sum()
logs_clean = logs_clean.dropna()
nan_removed = nan_before - logs_clean.isnull().sum().sum()
print(f"✅ NaN supprimés: {nan_removed}")

# 1.3 Réinitialiser l'index
logs_clean = logs_clean.reset_index(drop=True)

print(f"\n📊 Après nettoyage:")
print(f"   - Logs: {len(logs_clean)} lignes")

# ============================================
# 2. DISTRIBUTION DES INTÉRÊTS (Test χ²) - CORRIGÉ
# ============================================
print("\n\n2️⃣  DISTRIBUTION DES INTÉRÊTS")
print("-"*40)

# Compter les occurrences de chaque intérêt
all_interests = []
for interests_list in users_df['interests']:
    all_interests.extend(interests_list)

observed_counts = []
for interest in INTERESTS:
    observed_counts.append(all_interests.count(interest))

print(f"📊 Fréquences observées (sur {len(users_df)} utilisateurs):")
for interest, count in zip(INTERESTS, observed_counts):
    pct = (count / len(users_df)) * 100
    print(f"   - {interest:12}: {count:3} occurrences ({pct:.1f}%)")

# Distribution théorique
total_observations = sum(observed_counts)
expected_counts = [total_observations * p for p in INTEREST_PROBS]

print(f"\n📐 Distribution théorique (basée sur {total_observations} intérêts):")
for interest, prob, expected in zip(INTERESTS, INTEREST_PROBS, expected_counts):
    print(f"   - {interest:12}: prob={prob:.0%} → {expected:.1f} attendus")

# Conversion en entiers et équilibrage
expected_counts_int = [int(round(x)) for x in expected_counts]
sum_expected = sum(expected_counts_int)
sum_observed = sum(observed_counts)

if sum_expected != sum_observed:
    diff = sum_observed - sum_expected
    expected_counts_int[-1] += diff

# Test du χ²
chi2_interests, p_value_interests = stats.chisquare(observed_counts, expected_counts_int)

print(f"\n📈 Résultat du test χ²:")
print(f"   - χ² = {chi2_interests:.4f}")
print(f"   - p-value = {p_value_interests:.6f}")

if p_value_interests < 0.05:
    print(f"   ✅ Différence SIGNIFICATIVE")
else:
    print(f"   ❌ Pas de différence significative")

# ============================================
# 3. ANALYSE DES TYPES D'ACTIONS
# ============================================
print("\n\n3️⃣  ANALYSE DES TYPES D'ACTIONS (view/buy/like)")
print("-"*40)

action_type_counts = logs_clean['action_type'].value_counts()
print(f"📊 Distribution des actions par type:")
for action_type, count in action_type_counts.items():
    pct = (count / len(logs_clean)) * 100
    print(f"   - {action_type:8}: {count:5} ({pct:.1f}%)")

# Test χ²
chi2_type, p_value_type = stats.chisquare(action_type_counts)
print(f"\n📈 Test χ² (uniformité):")
print(f"   - χ² = {chi2_type:.4f}")
print(f"   - p-value = {p_value_type:.6f}")

# ============================================
# 4. CORRÉLATION ACTIVITÉS
# ============================================
print("\n\n4️⃣  CORRÉLATION ENTRE ACTIVITÉS")
print("-"*40)

# Créer un tableau user × activité
user_actions = logs_clean.groupby('user_id')['action'].apply(list).reset_index()

# Corrélation: "joined gym" vs "bought protein powder"
user_actions['joined_gym'] = user_actions['action'].apply(
    lambda x: 1 if 'joined gym' in x else 0
)
user_actions['bought_protein'] = user_actions['action'].apply(
    lambda x: 1 if 'bought protein powder' in x else 0
)

contingency_gym = pd.crosstab(user_actions['joined_gym'], user_actions['bought_protein'])
print(f"\n🔍 'joined gym' vs 'bought protein powder'")
print(contingency_gym)

if contingency_gym.shape == (2,2):
    chi2_gym, p_gym, dof_gym, expected_gym = stats.chi2_contingency(contingency_gym)
    print(f"\n   χ² = {chi2_gym:.4f}, p-value = {p_gym:.4f}")
    if p_gym < 0.05:
        print(f"   ✅ Corrélation SIGNIFICATIVE")
    else:
        print(f"   ❌ Pas de corrélation significative")

# ============================================
# 5. SIMULATION DE DISTRIBUTIONS
# ============================================
print("\n\n5️⃣  SIMULATION DE DISTRIBUTIONS DE PROBABILITÉ")
print("-"*40)

n_sim = 1000
simulated_uniform = np.random.choice(INTERESTS, size=n_sim, p=[1/6]*6)
unique, uniform_counts = np.unique(simulated_uniform, return_counts=True)

print(f"📊 Simulation uniforme (1000 utilisateurs):")
for interest, count in zip(unique, uniform_counts):
    print(f"   - {interest:12}: {count:3} ({count/n_sim:.0%})")

# ============================================
# 6. ANALYSE TEMPORELLE
# ============================================
print("\n\n6️⃣  ANALYSE TEMPORELLE")
print("-"*40)

hourly_activity = logs_clean.groupby('hour').size()
peak_hour = hourly_activity.idxmax()
print(f"📊 Heure de pointe: {peak_hour}h ({hourly_activity.max()} activités)")

# Test de normalité
if len(hourly_activity) >= 3:
    shapiro_stat, shapiro_p = stats.shapiro(hourly_activity)
    print(f"\n📈 Test de normalité (Shapiro-Wilk): p={shapiro_p:.4f}")

# ============================================
# RÉSUMÉ
# ============================================
print("\n" + "="*60)
print("🎉 ÉTAPE 2 TERMINÉE !")
print("="*60)

=== ÉTAPE 2 : PRÉTRAITEMENT ET ANALYSE STATISTIQUE ===

1️⃣  NETTOYAGE DES DONNÉES
----------------------------------------
📊 Avant nettoyage:
   - Logs: 5193 lignes
   - Utilisateurs: 500 lignes

✅ Doublons supprimés: 138
✅ NaN supprimés: 103

📊 Après nettoyage:
   - Logs: 4952 lignes


2️⃣  DISTRIBUTION DES INTÉRÊTS
----------------------------------------
📊 Fréquences observées (sur 500 utilisateurs):
   - fitness     : 364 occurrences (72.8%)
   - musique     : 318 occurrences (63.6%)
   - technologie : 321 occurrences (64.2%)
   - cuisine     : 287 occurrences (57.4%)
   - voyage      : 281 occurrences (56.2%)
   - cinema      : 207 occurrences (41.4%)

📐 Distribution théorique (basée sur 1778 intérêts):
   - fitness     : prob=25% → 444.5 attendus
   - musique     : prob=22% → 391.2 attendus
   - technologie : prob=18% → 320.0 attendus
   - cuisine     : prob=15% → 266.7 attendus
   - voyage      : prob=12% → 213.4 attendus
   - cinema      : prob=8% → 142.2 attendus

📈 Résultat 